# SAM3 — batch segment a folder of tiles

Segments **every** image in an input folder with a SAM3 text prompt (**instance** segmentation), then for each image saves:

- `tiles_sam/<name>_mask.png` — labelled mask where **each seedling has a unique integer value** (0 = background). Saved as 16-bit PNG so it holds >255 instances.
- `tiles_sam/<name>_overlay.png` — coloured overlay where **each plant gets its own colour**.

Finally it builds an **animation** that cycles, per image, through: **raw → overlay → mask**, saved as `tiles_sam/animation.gif` (and `.mp4` if ffmpeg is available).

In [15]:
# Cell 1 — environment check
import torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")
    print("CUDA:", torch.version.cuda)

PyTorch: 2.12.0+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 5090
VRAM: 34.19045888 GB
CUDA: 13.0


In [16]:
# Cell 2 — load model
import sys, os
sys.path.insert(0, "..")

import torch
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CHECKPOINT = "../checkpoints/sam3.pt"

model = build_sam3_image_model(
    checkpoint_path=CHECKPOINT,
    load_from_HF=False,
    device=DEVICE,
    eval_mode=True,
).to(DEVICE)

processor = Sam3Processor(model, confidence_threshold=0.3)
print("Model loaded OK")

Model loaded OK


In [17]:
# Cell 3 — configuration
import os

# Windows path D:\SIF_CAL_HIRES\T1_proc\tiles maps to this WSL path:
INPUT_DIR  = "/mnt/d/SIF_CAL_HIRES/T1_proc/tiles"
OUTPUT_DIR = "/mnt/d/SIF_CAL_HIRES/T1_proc/tiles_sam"

PROMPT          = "seedling"   # text prompt fed to SAM3
SCORE_THRESHOLD = 0.5          # keep instances at/above this confidence
IMG_EXTS        = (".tif", ".tiff", ".png", ".jpg", ".jpeg", ".bmp")

OVERLAY_ALPHA   = 0.65                  # per-instance mask opacity in overlay (each plant a different colour)

ANIM_FRAME_MS   = 600   # ms per frame in the animation

os.makedirs(OUTPUT_DIR, exist_ok=True)

image_files = sorted(
    f for f in os.listdir(INPUT_DIR)
    if f.lower().endswith(IMG_EXTS)
)
print(f"Input : {INPUT_DIR}")
print(f"Output: {OUTPUT_DIR}")
print(f"Found {len(image_files)} image(s)")

Input : /mnt/d/SIF_CAL_HIRES/T1_proc/tiles
Output: /mnt/d/SIF_CAL_HIRES/T1_proc/tiles_sam
Found 100 image(s)


In [18]:
# Cell 4 — helpers
import json
import numpy as np
from PIL import Image
import colorsys

try:
    import cv2
except ImportError:
    import subprocess, sys
    print("Installing opencv-python-headless ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "opencv-python-headless"], check=True)
    import cv2


def segment_instances(pil_img, prompt, score_threshold):
    """Run SAM3 once; return a labelled mask (uint16) where each kept instance
    has a unique value 1..N (0 = background), the instance count, and the list
    of kept confidence scores (ordered by label 1..N).
    Later (higher-scoring) instances win where masks overlap."""
    with torch.inference_mode():
        autocast = (
            torch.autocast(device_type="cuda", dtype=torch.bfloat16)
            if DEVICE == "cuda"
            else torch.autocast(device_type="cpu", enabled=False)
        )
        with autocast:
            state = processor.set_image(pil_img)
            out = processor.set_text_prompt(state=state, prompt=prompt)

    masks, scores = out["masks"], out["scores"]
    h, w = pil_img.size[1], pil_img.size[0]
    label_mask = np.zeros((h, w), dtype=np.uint16)

    # keep + sort by score ascending so the most confident instance is painted last
    kept = []
    for mask, score in zip(masks, scores):
        if float(score) < score_threshold:
            continue
        m = mask.cpu().numpy() if hasattr(mask, "cpu") else np.array(mask)
        kept.append((float(score), m.squeeze() > 0.5))
    kept.sort(key=lambda t: t[0])

    kept_scores = []
    for label, (score, mbool) in enumerate(kept, start=1):
        label_mask[mbool] = label
        kept_scores.append(score)
    return label_mask, len(kept), kept_scores


def _label_colour(label, n_total):
    """Deterministic distinct RGB (0-1) for an instance label using HSV spacing."""
    if label == 0:
        return np.array([0.0, 0.0, 0.0])
    hue = ((label - 1) * 0.61803398875) % 1.0   # golden-ratio hue spacing
    return np.array(colorsys.hsv_to_rgb(hue, 0.85, 1.0))


def make_instance_overlay(image_np, label_mask, alpha):
    """Blend a uniquely-coloured mask per instance onto the original. uint8 RGB."""
    overlay = image_np.astype(float) / 255.0
    labels = np.unique(label_mask)
    labels = labels[labels != 0]
    n = len(labels)
    for lab in labels:
        sel = label_mask == lab
        col = _label_colour(int(lab), n)
        overlay[sel] = overlay[sel] * (1 - alpha) + col * alpha
    return (overlay * 255).astype(np.uint8)


def colorize_labels(label_mask):
    """Render a labelled mask as a distinct-colour RGB image (for the animation). uint8."""
    h, w = label_mask.shape
    rgb = np.zeros((h, w, 3), dtype=float)
    labels = np.unique(label_mask)
    labels = labels[labels != 0]
    n = len(labels)
    for lab in labels:
        rgb[label_mask == lab] = _label_colour(int(lab), n)
    return (rgb * 255).astype(np.uint8)


def mask_to_polygons(bool_mask, min_area=10.0, approx_eps_frac=0.002):
    """Convert a boolean mask to a list of polygons (each a list of [x, y] points).
    Contours are simplified with Douglas-Peucker; tiny blobs are dropped."""
    mask_u8 = bool_mask.astype(np.uint8)
    contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    polys = []
    for cnt in contours:
        if cv2.contourArea(cnt) < min_area:
            continue
        eps = approx_eps_frac * cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, eps, True)
        if len(approx) < 3:
            continue
        polys.append(approx.reshape(-1, 2).astype(float).tolist())
    return polys


def build_anylabeling_shapes(label_mask, label):
    """Build AnyLabeling/LabelMe `shapes` from a labelled mask. Each instance keeps
    its label value as group_id so overlapping parts stay grouped."""
    shapes = []
    labels = np.unique(label_mask)
    labels = labels[labels != 0]
    for lab in labels:
        for poly in mask_to_polygons(label_mask == lab):
            shapes.append({
                "label": label,
                "points": poly,
                "group_id": int(lab),
                "description": "",
                "difficult": False,
                "shape_type": "polygon",
                "flags": {},
                "attributes": {},
            })
    return shapes


def save_anylabeling_json(json_path, image_path, height, width, shapes):
    """Write an AnyLabeling/LabelMe JSON next to (or alongside) the image so the
    polygons can be opened and edited in AnyLabeling. imagePath is stored relative
    to the JSON location so it resolves when loaded."""
    rel_image = os.path.relpath(image_path, os.path.dirname(json_path))
    data = {
        "version": "2.4.0",
        "flags": {},
        "shapes": shapes,
        "imagePath": rel_image,
        "imageData": None,
        "imageHeight": int(height),
        "imageWidth": int(width),
    }
    with open(json_path, "w") as f:
        json.dump(data, f, indent=2)

In [19]:
# Cell 5 — batch segment every image (instance segmentation)
import time

results = []   # (name, raw_uint8, overlay_uint8, label_mask_uint16)
t0 = time.time()

for i, fname in enumerate(image_files):
    stem = os.path.splitext(fname)[0]
    img = Image.open(os.path.join(INPUT_DIR, fname)).convert("RGB")
    image_np = np.array(img)

    label_mask, n_kept, kept_scores = segment_instances(img, PROMPT, SCORE_THRESHOLD)
    overlay_np = make_instance_overlay(image_np, label_mask, OVERLAY_ALPHA)

    Image.fromarray(overlay_np).save(os.path.join(OUTPUT_DIR, f"{stem}_overlay.png"))

    # editable polygons for AnyLabeling (LabelMe-style JSON next to the original tile)
    shapes = build_anylabeling_shapes(label_mask, PROMPT)
    save_anylabeling_json(
        os.path.join(OUTPUT_DIR, f"{stem}.json"),
        os.path.join(INPUT_DIR, fname),
        image_np.shape[0], image_np.shape[1],
        shapes,
    )

    coverage = 100.0 * np.count_nonzero(label_mask) / label_mask.size
    results.append((stem, image_np, overlay_np, label_mask))
    print(f"[{i+1:>4}/{len(image_files)}] {fname:40s} {n_kept:3d} instance(s)  "
          f"{len(shapes):3d} polygon(s)  "
          f"{coverage:5.1f}% coverage  [{time.time()-t0:6.1f}s]")
    # per-instance confidence (label 1..N, matching the labelled mask values)
    for label, score in enumerate(kept_scores, start=1):
        print(f"         instance {label:3d}: {score:.3f} confidence")

print(f"\nDone. Saved overlays + AnyLabeling JSON for {len(results)} image(s) to {OUTPUT_DIR}")

[   1/100] P0063102_tile_c4_r8.tif                    8 instance(s)   10 polygon(s)    1.2% coverage  [   0.4s]
         instance   1: 0.504 confidence
         instance   2: 0.504 confidence
         instance   3: 0.539 confidence
         instance   4: 0.562 confidence
         instance   5: 0.566 confidence
         instance   6: 0.602 confidence
         instance   7: 0.621 confidence
         instance   8: 0.695 confidence
[   2/100] P0063116_tile_c8_r0.tif                   18 instance(s)   18 polygon(s)    1.6% coverage  [   0.7s]
         instance   1: 0.512 confidence
         instance   2: 0.523 confidence
         instance   3: 0.535 confidence
         instance   4: 0.586 confidence
         instance   5: 0.594 confidence
         instance   6: 0.609 confidence
         instance   7: 0.629 confidence
         instance   8: 0.656 confidence
         instance   9: 0.676 confidence
         instance  10: 0.680 confidence
         instance  11: 0.711 confidence
         instanc

In [20]:
# Cell 6 — build animation: per image -> raw, overlay  (MP4 output)
import numpy as np
from PIL import Image

frames = []
for stem, raw, overlay, label_mask in results:
    for img_np in (raw, overlay):   # raw -> overlay only (no mask frame)
        frames.append(Image.fromarray(img_np).convert("RGB"))

if not frames:
    raise RuntimeError("No frames to animate — run the batch cell first.")

# Normalise all frames to the first frame's size, and make dims even (H.264 needs even W/H)
base_w, base_h = frames[0].size
base_w -= base_w % 2
base_h -= base_h % 2
frames = [f.resize((base_w, base_h)) for f in frames]

# Ensure the ffmpeg backend is available
try:
    import imageio_ffmpeg  # noqa: F401
except ImportError:
    import subprocess, sys
    print("Installing imageio-ffmpeg ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "imageio-ffmpeg"], check=True)

import imageio.v2 as imageio

mp4_path = os.path.join(OUTPUT_DIR, "animation.mp4")
fps = max(1, round(1000 / ANIM_FRAME_MS))

writer = imageio.get_writer(
    mp4_path,
    format="FFMPEG",
    mode="I",
    fps=fps,
    codec="libx264",
    quality=8,
    macro_block_size=None,
)
for f in frames:
    writer.append_data(np.array(f))
writer.close()

print(f"Saved MP4 ({len(frames)} frames @ {fps} fps) → {mp4_path}")

[rawvideo @ 0x25b310c0] Stream #0: not enough frames to estimate rate; consider increasing probesize


Saved MP4 (200 frames @ 2 fps) → /mnt/d/SIF_CAL_HIRES/T1_proc/tiles_sam/animation.mp4


In [21]:
# # Cell 7 — preview the animation inline
# from IPython.display import Image as IPyImage, display
# display(IPyImage(filename=os.path.join(OUTPUT_DIR, "animation.gif")))